<a href="https://colab.research.google.com/github/aleynakirmizi/DI_725_Assignment_1/blob/main/fixed_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **WANDB Connection**

In [ ]:
!pip install wandb

In [ ]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: ERROR Invalid API key: API key must have 40+ characters, has 36.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

...

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: e274020 (e274020-middle-east-technical-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
wandb.init(
    project="sentiment-analysis-assignment",
    name="roberta-class-weighted"
)


# **Libraries**

In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import plotly.express as px
import plotly.graph_objects as go

import torch
from torch import nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from transformers import RobertaTokenizer, RobertaForSequenceClassification, Trainer, TrainingArguments

warnings.filterwarnings('ignore')

# **Exploratory Data Analysis**

In [ ]:
df_train = pd.read_csv('train.csv')
df_test = pd.read_csv('test.csv')

In [ ]:
print("df_train shape", df_train.shape)
print("df_test shape", df_test.shape)

df_train shape (970, 11)
df_test shape (30, 11)


In [ ]:
df_train.isnull().sum()

,0
issue_area,0
issue_category,0
issue_sub_category,0
issue_category_sub_category,0
customer_sentiment,0
product_category,0
product_sub_category,0
issue_complexity,0
agent_experience_level,0
agent_experience_level_desc,0


In [ ]:
df_test.isnull().sum()

,0
issue_area,0
issue_category,0
issue_sub_category,0
issue_category_sub_category,0
customer_sentiment,0
product_category,0
product_sub_category,0
issue_complexity,0
agent_experience_level,0
agent_experience_level_desc,0


In [ ]:
df_train.head(3)

,issue_area,issue_category,issue_sub_category,issue_category_sub_category,customer_sentiment,product_category,product_sub_category,issue_complexity,agent_experience_level,agent_experience_level_desc,conversation
0,Login and Account,Mobile Number and Email Verification,Verification requirement for mobile number or ...,Mobile Number and Email Verification -> Verifi...,neutral,Appliances,Oven Toaster Grills (OTG),medium,junior,"handles customer inquiries independently, poss...",Agent: Thank you for calling BrownBox Customer...
1,Cancellations and returns,Pickup and Shipping,Reasons for being asked to ship the item,Pickup and Shipping -> Reasons for being asked...,neutral,Electronics,Computer Monitor,less,junior,"handles customer inquiries independently, poss...",Agent: Thank you for calling BrownBox customer...
2,Cancellations and returns,Replacement and Return Process,Inability to click the 'Cancel' button,Replacement and Return Process -> Inability to...,neutral,Appliances,Juicer/Mixer/Grinder,medium,experienced,"confidently handles complex customer issues, e...",Agent: Thank you for calling BrownBox Customer...


In [ ]:
df_train.columns

Index(['issue_area', 'issue_category', 'issue_sub_category',
       'issue_category_sub_category', 'customer_sentiment', 'product_category',
       'product_sub_category', 'issue_complexity', 'agent_experience_level',
       'agent_experience_level_desc', 'conversation'],
      dtype='object')

In [ ]:
df_train['customer_sentiment'].value_counts()

,count
customer_sentiment,
neutral,542
negative,411
positive,17


In [ ]:
pd.crosstab(df_train["issue_area"], df_train["customer_sentiment"])

customer_sentiment,negative,neutral,positive
issue_area,,,
Cancellations and returns,136,141,0
Login and Account,28,121,0
Order,156,85,17
Shipping,37,33,0
Shopping,36,77,0
Warranty,18,85,0


In [ ]:
sentiment_counts = df_train['customer_sentiment'].value_counts().reset_index()
sentiment_counts.columns = ['customer_sentiment', 'count']

fig = px.pie(
    sentiment_counts,
    names='customer_sentiment',
    values='count',
    title='Customer Sentiment Distribution'
)
fig.show()

In [ ]:
df_train['word_count'] = df_train['conversation'].apply(lambda x: len(str(x).split()))
df_train['char_length'] = df_train['conversation'].str.len()

# 1. Text Length Distribution (Histogram)
fig1 = px.histogram(df_train, x='word_count', nbins=50,
                   title='Konuşma Uzunluğu (Kelime Sayısı) Dağılımı',
                   labels={'word_count': 'Kelime Sayısı'},
                   color_discrete_sequence=['#636EFA'])
fig1.show()

# 2. The Relationship Between Emotion and Text Length (Box Plot)
# Answers the question: ‘Are negative comments longer?’
fig2 = px.box(df_train, x='customer_sentiment', y='word_count',
             color='customer_sentiment',
             title='Duygu Sınıflarına Göre Kelime Sayısı Dağılımı',
             labels={'customer_sentiment': 'Duygu', 'word_count': 'Kelime Sayısı'})
fig2.show()

# 3. Issue Area and Sentiment Correlation (Stacked Bar)
# Which departments have higher levels of negative sentiment?
issue_sentiment = df_train.groupby(['issue_area', 'customer_sentiment']).size().reset_index(name='count')
fig3 = px.bar(issue_sentiment, x='issue_area', y='count', color='customer_sentiment',
             title='Departman Bazlı Duygu Dağılımı',
             labels={'issue_area': 'Konu Alanı', 'count': 'Adet'},
             barmode='stack')
fig3.update_layout(xaxis={'categoryorder':'total descending'})
fig3.show()

issue_complexity = df_train.groupby(['issue_complexity', 'customer_sentiment']).size().reset_index(name='count')
fig4 = px.bar(issue_complexity, x='issue_complexity', y='count', color='customer_sentiment',
             title='Distribution of Customer Sentiment over Issue Complexity',
             labels={'issue_complexity': 'Issue Complexity', 'count': 'Count'},
             barmode='stack')
fig4.update_layout(xaxis={'categoryorder':'total descending'})
fig4.show()

# **Cleaning**

Reducing the number of token because the models has limited token numbers.

In [ ]:
df_train["conversation"][20]

"Agent: Thank you for calling BrownBox Customer Support. My name is Sarah. How can I assist you today?\n\nCustomer: Hi Sarah, I'm John. I have an issue with my recent order.\n\nAgent: Hi John, I'm here to help you. Can you please provide me with your order number?\n\nCustomer: Sure, my order number is BB123456789.\n\nAgent: Thank you for providing that information, John. I see that you ordered a pair of shorts on May 10th. How can I assist you with it?\n\nCustomer: I'm very disappointed because I haven't received my package yet. It's been two weeks since I placed the order, and I haven't received any updates on the shipping. I was wondering if you could provide me with a tracking number or any information about the status of my package.\n\nAgent: I'm sorry to hear that, John. Let me check the status of your order for you. Please hold on for a moment while I access the information.\n\n(Customer is put on hold for a minute)\n\nAgent: Thank you for waiting, John. I have checked the status

In [ ]:
df_train["conversation"][250]

"Agent: Thank you for calling BrownBox Customer Support. My name is Sarah. How may I assist you today?\n\nCustomer: Hi Sarah, this is John. I have a question regarding a sweatshirt I purchased from your website.\n\nAgent: Sure, John. I'd be happy to help you with that. Can you please provide me with your order number or email address associated with the purchase?\n\nCustomer: Yes, my order number is BB987654321.\n\nAgent: Thank you, John. I see that you ordered a sweatshirt from one of our sellers. What can I help you with?\n\nCustomer: I was wondering if you could help me find the seller's return policy for this sweatshirt. I'm not able to find it on the website.\n\nAgent: I understand, John. Let me check that for you. Can you please provide me with the name of the seller?\n\nCustomer: Yes, the seller's name is FashionableWear.\n\nAgent: Thank you, John. Let me check the seller's return policy for you.\n\n[Agent puts the customer on hold briefly to investigate the issue]\n\nAgent: Tha

In [ ]:
def clean_intro(text):
    text = text.lower()
    pattern = r"agent:\s*(?:hello|hi)?[.,]?\s*thank you for (?:calling|contacting) brownbox customer support[.,]?\s*my name is .*?\.\s*how (?:can|may) i assist you today\?\s*customer:\s*"
    cleaned_text = re.sub(pattern, "", text)
    return cleaned_text.strip()

def clean_newlines(text):
  text= text.replace('\n',' ')
  text = re.sub(r'\s+', ' ', text)
  return text.strip()

def clean_system_notes(text):
  cleaned_text = re.sub(r'\s*\(.*?\)\s*', ' ', text)
  cleaned_text = re.sub(r'\s+', ' ', cleaned_text)
  cleaned_text = re.sub(r'\[.*?\]', '', cleaned_text)
  return cleaned_text.strip()


df_train['conversation'] = df_train['conversation'].apply(clean_intro)
df_train['conversation'] = df_train['conversation'].apply(clean_newlines)
df_train['conversation'] = df_train['conversation'].apply(clean_system_notes)

In [ ]:
df_train["conversation"][250]

"hi sarah, this is john. i have a question regarding a sweatshirt i purchased from your website. agent: sure, john. i'd be happy to help you with that. can you please provide me with your order number or email address associated with the purchase? customer: yes, my order number is bb987654321. agent: thank you, john. i see that you ordered a sweatshirt from one of our sellers. what can i help you with? customer: i was wondering if you could help me find the seller's return policy for this sweatshirt. i'm not able to find it on the website. agent: i understand, john. let me check that for you. can you please provide me with the name of the seller? customer: yes, the seller's name is fashionablewear. agent: thank you, john. let me check the seller's return policy for you.  agent: thank you for waiting, john. i've reviewed the seller's return policy, and i see that they offer a 30-day return window for their products. however, the sweatshirt you purchased is marked as final sale, which me

# **Class Weights**

In [ ]:
df_train["customer_sentiment"].unique()[1]

'negative'

In [ ]:
label_map = {label: idx for idx, label in enumerate(df_train["customer_sentiment"].unique())}
label_map

{'neutral': 0, 'negative': 1, 'positive': 2}

In [ ]:
df_train['label'] = df_train['customer_sentiment'].map(label_map)
df_train.head(2)

,issue_area,issue_category,issue_sub_category,issue_category_sub_category,customer_sentiment,product_category,product_sub_category,issue_complexity,agent_experience_level,agent_experience_level_desc,conversation,label
0,Login and Account,Mobile Number and Email Verification,Verification requirement for mobile number or ...,Mobile Number and Email Verification -> Verifi...,neutral,Appliances,Oven Toaster Grills (OTG),medium,junior,"handles customer inquiries independently, poss...","hi tom, i'm trying to log in to my account to ...",0
1,Cancellations and returns,Pickup and Shipping,Reasons for being asked to ship the item,Pickup and Shipping -> Reasons for being asked...,neutral,Electronics,Computer Monitor,less,junior,"handles customer inquiries independently, poss...",hi alex. i recently received an email from bro...,0


In [ ]:
label_test = {label: idx for idx, label in enumerate(df_train["customer_sentiment"].unique())}
df_test['label'] = df_test['customer_sentiment'].map(label_test)
df_test.head(2)

,issue_area,issue_category,issue_sub_category,issue_category_sub_category,customer_sentiment,product_category,product_sub_category,issue_complexity,agent_experience_level,agent_experience_level_desc,conversation,label
0,Shopping,Pricing and Discounts,Discounts through exchange offers,Pricing and Discounts -> Discounts through exc...,negative,Appliances,Hand Blender,less,experienced,"confidently handles complex customer issues, e...",Agent: Thank you for calling BrownBox Customer...,1
1,Login and Account,Account Reactivation and Deactivation,Reactivating an inactive account,Account Reactivation and Deactivation -> React...,negative,Men/Women/Kids,Wrist Watch,medium,junior,"handles customer inquiries independently, poss...",Agent: Thank you for calling BrownBox Customer...,1


In [ ]:
label_test

{0: 'negative', 1: 'neutral', 2: 'positive'}

# **Roberta**

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(df_train["label"]),
    y=df_train["label"]
)

class_weights = torch.tensor(class_weights, dtype=torch.float)
class_weights


tensor([ 0.5966,  0.7867, 19.0196])

In [ ]:
from transformers import AutoTokenizer # Added this import

tokenizer = AutoTokenizer.from_pretrained("roberta-base")

def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

In [ ]:
train_df = df_train[["conversation", "label"]]
test_df = df_test[["conversation"]]

In [ ]:
train_df, val_df = train_test_split(
    df_train,
    test_size=0.2,
    stratify=df_train["label"],
    random_state=42
)

In [ ]:
from datasets import Dataset

# Rename 'conversation' column to 'text' for tokenization
train_df_processed = train_df.rename(columns={'conversation': 'text'})
val_df_processed = val_df.rename(columns={'conversation': 'text'})
df_test_processed = df_test.rename(columns={'conversation': 'text'})

train_dataset = Dataset.from_pandas(train_df_processed)
val_dataset = Dataset.from_pandas(val_df_processed)
test_dataset = Dataset.from_pandas(df_test_processed)

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
val_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask"])

Map:   0%|          | 0/776 [00:00<?, ? examples/s]

Map:   0%|          | 0/194 [00:00<?, ? examples/s]

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

In [ ]:
from transformers import AutoModelForSequenceClassification # Added this import

model = AutoModelForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=3
)

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
class_weights = class_weights.to(device)

loss_fn = CrossEntropyLoss(weight=class_weights)
class CustomTrainer(HFTrainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None): # Added num_items_in_batch
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss = loss_fn(logits, labels)

        return (loss, outputs) if return_outputs else loss

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="weighted"
    )
    acc = accuracy_score(labels, preds)

    wandb.log({
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall,
        "confusion_matrix": wandb.plot.confusion_matrix(
            probs=None,
            y_true=labels,
            preds=preds,
            class_names=["neg", "neu", "pos"]
        )
    })

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }


In [ ]:
training_args = TrainingArguments(
    output_dir="./results",

    num_train_epochs=10,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="f1",

    logging_strategy="steps",
    logging_steps=50,

    report_to="wandb",
    run_name="roberta-sentiment-analysis",

    warmup_ratio=0.1,
    seed=42
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [ ]:
from transformers import EarlyStoppingCallback
trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,No log,0.374584,0.865979,0.866311,0.867395,0.865979
2,0.076984,0.481613,0.886598,0.887054,0.890584,0.886598
3,0.083038,0.565160,0.881443,0.882359,0.888597,0.881443
4,0.017492,0.776621,0.891753,0.890193,0.893802,0.891753
5,0.024914,0.495138,0.902062,0.901630,0.902456,0.902062
6,0.017378,0.486258,0.912371,0.912304,0.912288,0.912371
7,0.002388,0.635230,0.881443,0.881673,0.882336,0.881443
8,0.008283,0.638726,0.891753,0.891829,0.891954,0.891753


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=392, training_loss=0.029450600320587352, metrics={'train_runtime': 60.3274, 'train_samples_per_second': 128.632, 'train_steps_per_second': 8.122, 'total_flos': 408352024313856.0, 'train_loss': 0.029450600320587352, 'epoch': 8.0})

In [ ]:
predictions = trainer.predict(test_dataset)

logits = predictions.predictions
labels = predictions.label_ids

if labels is None and 'label' in df_test_processed.columns:
    labels = df_test_processed['label'].values
elif labels is None and 'label' not in df_test_processed.columns:
    print("Error: True labels are not available in predictions or df_test_processed. Cannot compute metrics.")
    raise ValueError("True labels are not available for metric computation.")


preds = np.argmax(logits, axis=1)

precision, recall, f1, _ = precision_recall_fscore_support(
    labels, preds, average="weighted"
)
acc = accuracy_score(labels, preds)

wandb.log({
    "test_accuracy": acc,
    "test_f1": f1
})

print("Test Accuracy:", acc)
print("Test F1:", f1)

Test Accuracy: 0.8333333333333334
Test F1: 0.8353793691389598
